# Streetsign Model

In this file the classifier and detector trained in the respective files are loaded and combined to a pipline, so that end-to-end image detection with classification is possible

## Load models and data

In [2]:
import torch
from ultralytics import YOLO
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision import transforms
from PIL import Image
import numpy as np

In [3]:
# Load your trained YOLOv8 model
yolo_model = YOLO("runs/detect/yolov8s_mapillary_subset_cpu10/weights/best.pt")


In [4]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision.datasets import ImageFolder

train_dir = "crops_per_label"

dataset = ImageFolder(train_dir)
class_names = dataset.classes   # SORTED and consistent
num_classes = len(class_names)
class_names = dataset.classes   # SORTED and consistent

weights = EfficientNet_B0_Weights.DEFAULT
classifier = efficientnet_b0(weights=weights)
classifier.classifier[1] = torch.nn.Linear(
    classifier.classifier[1].in_features,
    num_classes
)

classifier.load_state_dict(
    torch.load("street_sign_classifier_weights.pth", map_location="cpu")
)
classifier.eval()

classifier_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


## Method that runs whole pipeline

In [5]:
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import torch


def detect_and_classify(image_path, conf_threshold=0.3, show_image=False):
    """
    Input: full image path
    Output: list of dicts with bbox + predicted label
    """
    image = Image.open(image_path).convert("RGB")
    image_np = np.array(image)

    # YOLO detection
    results = yolo_model(image_np, conf=conf_threshold)[0]

    outputs = []

    if results.boxes is None:
        return outputs

    for box in results.boxes:
        # Bounding box
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        det_conf = float(box.conf[0])

        # Crop
        crop = image.crop((x1, y1, x2, y2))

        # Classifier inference
        input_tensor = classifier_transform(crop).unsqueeze(0)

        with torch.no_grad():
            logits = classifier(input_tensor)
            probs = torch.softmax(logits, dim=1)
            cls_idx = probs.argmax(dim=1).item()
            cls_conf = probs[0, cls_idx].item()

        outputs.append({
            "bbox": [x1, y1, x2, y2],
            "det_conf": det_conf,
            "label": class_names[cls_idx],
            "cls_conf": cls_conf
        })

    # Visualization using PIL
    if show_image:
        draw = ImageDraw.Draw(image)

        try:
            font = ImageFont.truetype("arial.ttf", size=20)
        except IOError:
            font = ImageFont.load_default()

        for out in outputs:
            x1, y1, x2, y2 = out["bbox"]
            label = out["label"]
            score = out["cls_conf"]

            # Draw bounding box
            draw.rectangle(
                [(x1, y1), (x2, y2)],
                outline="red",
                width=3
            )

            text = f"{label} ({score:.2f})"

            # Text background
            text_bbox = draw.textbbox((x1, y1), text, font=font)
            draw.rectangle(text_bbox, fill="red")

            # Draw text
            draw.text(
                (x1, y1),
                text,
                fill="white",
                font=font
            )

        image.show()

    return outputs


## Inference

In [6]:
image_path ="dataset/images/val/0MvoaXqbynGMHz55vfbFQA.jpg"

predictions = detect_and_classify(image_path, show_image=True)

for p in predictions:
    print(
        f"BBox: {p['bbox']}, "
        f"Label: {p['label']}, "
        f"Detection conf: {p['det_conf']:.2f}, "
        f"Class conf: {p['cls_conf']:.2f}"
    )



0: 480x640 3 streetsigns, 91.3ms
Speed: 20.9ms preprocess, 91.3ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)
BBox: [1649, 1571, 1737, 1640], Label: other-sign, Detection conf: 0.75, Class conf: 0.52
BBox: [1663, 1648, 1735, 1700], Label: warning--curve-right--g1, Detection conf: 0.66, Class conf: 0.11
BBox: [1361, 1681, 1416, 1733], Label: information--pedestrians-crossing--g1, Detection conf: 0.61, Class conf: 0.95


In [7]:
import glob
import random

VAL_DIR = "dataset/images/val"

# Collect all images
image_paths = glob.glob(f"{VAL_DIR}/*.jpg") + glob.glob(f"{VAL_DIR}/*.png")
assert len(image_paths) > 0, "No images found in val directory!"

# Pick one random image
image_path = random.choice(image_paths)

print(f"Selected image: {image_path}")

# Run pipeline
predictions = detect_and_classify(image_path, show_image=True)

# Print results
for p in predictions:
    print(
        f"BBox: {p['bbox']}, "
        f"Label: {p['label']}, "
        f"Detection conf: {p['det_conf']:.2f}, "
        f"Class conf: {p['cls_conf']:.2f}"
    )


Selected image: dataset/images/val\60mrthmJcjn78K9xYXNSNw.jpg

0: 480x640 2 streetsigns, 77.8ms
Speed: 1.8ms preprocess, 77.8ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)
BBox: [1838, 738, 1991, 894], Label: other-sign, Detection conf: 0.80, Class conf: 0.08
BBox: [441, 1175, 488, 1234], Label: other-sign, Detection conf: 0.45, Class conf: 0.96


## code for showcase video

In [12]:
image_path ="dataset/images/val/9DK08QYXpdoctONe0hnq1Q.jpg"

predictions = detect_and_classify(image_path, show_image=True)

for p in predictions:
    print(
        f"BBox: {p['bbox']}, "
        f"Label: {p['label']}, "
        f"Detection conf: {p['det_conf']:.2f}, "
        f"Class conf: {p['cls_conf']:.2f}"
    )


0: 480x640 1 streetsign, 83.3ms
Speed: 2.0ms preprocess, 83.3ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)
BBox: [88, 1542, 375, 1793], Label: regulatory--keep-right--g1, Detection conf: 0.91, Class conf: 0.32
